### Loading necessary dependencies

In [1]:
import cobra
import pickle
import numpy as np
import pandas as pd
import json
import cobrame
import ecolime

### Defining functions to solve the ME model

In [2]:
# Solving the model 
def solve_me_model(me, max_mu, precision=1e-6, min_mu=0, using_soplex=True,
                  compiled_expressions=None):
    if using_soplex:
        from cobrame.solve.algorithms import binary_search
        binary_search(me, min_mu=min_mu, max_mu=max_mu, debug=True, mu_accuracy=precision,
                      compiled_expressions=compiled_expressions)
    else:
        from qminospy.me1 import ME_NLP1
        # The object containing solveME methods--composite that uses a ME model object 
        me_nlp = ME_NLP1(me, growth_key='mu')
        # Use bisection for now (until the NLP formulation is worked out)
        muopt, hs, xopt, cache = me_nlp.bisectmu(precision=precision, mumax=max_mu)
        me.solution.f = me.solution.x_dict['biomass_dilution']
        

# Solving the model with a specific mu
def solve_me_model_fixed(me, muf, using_soplex=False):
    if using_soplex:
        from cobrame.solve.algorithms import solve_at_growth_rate
        solve_at_growth_rate(me, muf)
    else:
        from qminospy.me1 import ME_NLP1
        print('qminos')
        # The object containing solveME methods--composite that uses a ME model object 
        me_nlp = ME_NLP1(me, growth_key='mu')
        # Use solv for now
        xopt,status,hs = me_nlp.solvelp(0.69, verbosity=2)
        me.solution.f = me.solution.x_dict['biomass_dilution']

### Solving the model with every selected carbon source and exporting the solution as a dictionary

In [58]:
# Setting standard model parameters
upf = 0.36     # Unmodeled protein fraction
GAM = 34.98    # Growth associated maintenance energy
nGAM = 1       # non-Growth associated maintenance energy

# List with the uptake reactions of each selected carbon source
reactions = ['EX_glyc_e', 'EX_gal_e', 'EX_man_e', 'EX_gam_e', 'EX_pyr_e', 'EX_fru_e',
             'EX_glc__D_e', 'EX_xyl__D_e', 'EX_succ_e', 'EX_ac_e']

# Solving the model with each carbon source individually
for reaction in reactions:
    
    # Loading the model
    with open('iJL1678b_ML_KO_keff_FbFP.pickle', 'rb') as f:
        me = pickle.load(f)
    
    # Standard parameters
    me.gam = GAM
    me.ngam = nGAM
    me.unmodeled_protein_fraction = upf
    
    # Setting uptake fluxes
    me.reactions.get_by_id('EX_glc__D_e').lower_bound = 0 # Turning off glucose uptake
    me.reactions.get_by_id('EX_glc__D_e').upper_bound = 0
    
    me.reactions.get_by_id(reaction).lower_bound = -1000 # Turning on selected carbon source uptake
    me.reactions.get_by_id(reaction).upper_bound = 1000
    
    # Showing parameters before solving the model
    print('- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -')
    print(reaction)
    print('solving with\nUPF = ', str(me.unmodeled_protein_fraction))
    print('GAM', me.gam)
    print('nGAM', me.ngam)
    
    # Solving the model
    solve_me_model(me,
                   max_mu = 1.2,
                   min_mu = 0,
                   precision = 1e-6,
                   using_soplex = False)
    
    # Showing solution after solving the model
    print('\n')
    print(str(list(me.reactions.get_by_id(reaction).__dict__['_metabolites'].keys())[0]) + ' uptake'),
    print(me.solution.x_dict[reaction])
    print('ME growth rate'), print(me.solution.f)
    
    # Exporting the solution (with the reaction fluxes)
    with open('ME_Output/' + reaction + '_34_1_036.pkl', 'wb') as f:
        pickle.dump(me.solution.x_dict, f)

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
reaction
solving with
UPF =  0.36
GAM 34.98
nGAM 1
iter	muopt    	a     	b     	mu1       	stat1
Finished compiling expressions in 45.587933 seconds
Finished substituting S,lb,ub in 1.767664 seconds
Finished makeME_LP in 0.199804 seconds
Getting MINOS parameters from ME_NLP...
1 0.6 0.6 1.2 0.6 optimal
Finished substituting S,lb,ub in 1.776042 seconds
Finished makeME_LP in 0.202496 seconds
Getting MINOS parameters from ME_NLP...
2 0.8999999999999999 0.8999999999999999 1.2 0.8999999999999999 optimal
Finished substituting S,lb,ub in 1.761043 seconds
Finished makeME_LP in 0.198953 seconds
Getting MINOS parameters from ME_NLP...
3 0.8999999999999999 0.8999999999999999 1.0499999999999998 1.0499999999999998 1
Finished substituting S,lb,ub in 1.760421 seconds
Finished makeME_LP in 0.196787 seconds
Getting MINOS parameters from ME_NLP...
4 0.9749999999999999 0.974999999